In [33]:
from verimon import loaders
from verimon.logger import clear_logging, setup_logging
from stormpy import Rational

clear_logging()
setup_logging()

model_type = "SnL"
exact = True
learning_threshold = Rational(0.2)
unsafe_threshold = Rational(0.3)
safe_threshold = Rational(0.1)
horizon = 20

# POMDP settings
model_path = "../tests/premise/airportA-3.nm"
constants = "DMAX=4,PMAX=4"
pomdp_spec = 'Pmax=? [F<=2 "crash"]'

# Snake and Ladders settings
n, ladders, snakes = loaders.random_snl_board(3**2)
# milton_snakes = {
#     98: 76,
#     95: 75,
#     93: 73,
#     87: 24,
#     64: 60,
#     62: 19,
#     55: 53,
#     49: 11,
#     47: 26,
#     16: 6,
# }
# milton_ladders = {1: 38, 4: 14, 9: 31, 28: 64, 40: 42, 36: 44, 51: 67, 71: 91, 80: 100}
# n, ladders, snakes = 10**2, milton_ladders, milton_snakes

if model_type == "pomdp":
    (
        initial_observation,
        observations,
        mc,
        expr_manager,
    ) = loaders.pomdp_to_stormpy_mc(model_path, constants, exact)
    alphabet = list(observations)
    spec = pomdp_spec
elif model_type == "SnL":
    mc_sl_u_nxn = "../tests/snake_ladder/mc_u_nxn.pm"
    mc, expr_manager = loaders.load_snl_stormpy(
        mc_sl_u_nxn,
        n,
        ladders,
        snakes,
        exact,
    )
    alphabet = ["init", "normal", "snake", "ladder"]
    initial_observation = "init"
    spec = 'Pmax=? [F<5 "good"]'

n=9, l1s=4, l1d=6, l2s=2, l2d=4, l3s=3, l3d=3, l4s=-1, l4d=-1, l5s=-1, l5d=-1, l6s=-1, l6d=-1, l7s=-1, l7d=-1, l8d=-1, l8s=-1, l9s=-1, l9d=-1, l10s=-1, l10d=-1, s1s=6, s1d=3, s2s=8, s2d=2, s3s=-1, s3d=-1, s4s=-1, s4d=-1, s5s=-1, s5d=-1, s6s=-1, s6d=-1, s7s=-1, s7d=-1, s8d=-1, s8s=-1, s9s=-1, s9d=-1, s10s=-1, s10d=-1


In [34]:
from verimon.MonitorLearning import FilteringSUL


sul = FilteringSUL(
    mc,
    initial_observation,
    alphabet,
    spec,
    learning_threshold,
    horizon,
    True,
)

DEBUG:MainProcess:2025-09-12 15:09:40,947 - (0.00s) - MonitorLearning.py:102 - Filtering SUL is using the following risk function: [<Rational  (gmp)17/324>, <Rational  (gmp)13/162>, <Rational  (gmp)0>, <Rational  (gmp)0>, <Rational  (gmp)0>, <Rational  (gmp)20/81>, <Rational  (gmp)0>, <Rational  (gmp)20/81>, <Rational  (gmp)0>, <Rational  (gmp)1>] 


In [35]:
from tqdm import tqdm

words = []
good_prefixes = [[]]
for l in range(horizon + 1):
    print(f"Starting length {l}...")
    next_prefixes = []
    for pre in tqdm(good_prefixes):
        for a in alphabet:
            t = pre + [a]
            sul.pre()
            for obs in t:
                if sul.out_of_model:
                    break
                sul.step(obs)

            risk = sul.last_risk

            if sul.out_of_model:
                continue

            # print(f"Word: {t}, Risk: {float(risk)}")
            next_prefixes.append(t)

            if risk >= unsafe_threshold:
                words.append((t, "+"))
            elif risk <= safe_threshold:
                words.append((t, "-"))

    good_prefixes = next_prefixes

with open("words.txt", "w") as f:
    for w, label in words:
        f.write(f"{';'.join(w)}, {label}\n")

Starting length 0...


100%|██████████| 1/1 [00:00<00:00, 7307.15it/s]


Starting length 1...


100%|██████████| 3/3 [00:00<00:00, 22036.62it/s]


Starting length 2...


...

100%|██████████| 39/39 [00:00<00:00, 4382.88it/s]


Starting length 19...


100%|██████████| 41/41 [00:00<00:00, 4240.75it/s]


Starting length 20...


100%|██████████| 43/43 [00:00<00:00, 4047.01it/s]
